# 05 · P-M 상관도 — 공칭강도와 설계강도

원 문서의 `moment_interaction.ipynb` 에 대응한다.

| 항목 | 식 | 조문 |
|---|---|---|
| 순수압축 강도 | $P_o = 0.85f_{ck}(A_g-A_{st}) + f_yA_{st}$ | KDS 14 20 20 4.1.2(7) |
| 최대 설계 축강도 | $0.80\phi P_o$ (띠철근), $0.85\phi P_o$ (나선철근) | KDS 14 20 20 식 (4.1-16), (4.1-17) |
| 강도감소계수 | $\varepsilon_t$ 에 따라 0.65~0.85 | KDS 14 20 10 4.3.3(2) |

In [1]:
import matplotlib.pyplot as plt
import numpy as np

# 한글 글꼴이 없는 환경에서도 그림이 깨지지 않도록 축 라벨은 ASCII 로 둔다
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 96

In [2]:
from concreteproperties import ConcreteSection
from sectionproperties.pre.library import concrete_rectangular_section

from concreteproperties_kds import KDS


def beam_section(fck=27, fy=400):
    """400 x 600 보 단면 (상부 2-D16, 하부 4-D22, 피복 50 mm)."""
    kds = KDS(column_type="tie")
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=600, b=400,
        dia_top=16, area_top=198.6, n_top=2, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=4, c_bot=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec


def column_section(fck=27, fy=400, column_type="tie"):
    """500 x 500 기둥 단면 (8-D22, 피복 50 mm)."""
    kds = KDS(column_type=column_type)
    conc = kds.create_concrete_material(compressive_strength=fck)
    steel = kds.create_steel_material(yield_strength=fy)

    geom = concrete_rectangular_section(
        d=500, b=500,
        dia_top=22, area_top=387.1, n_top=3, c_top=50,
        dia_bot=22, area_bot=387.1, n_bot=3, c_bot=50,
        dia_side=22, area_side=387.1, n_side=1, c_side=50,
        n_circle=16, conc_mat=conc, steel_mat=steel,
    )
    conc_sec = ConcreteSection(geom)
    kds.assign_concrete_section(conc_sec)
    return kds, conc_sec

In [3]:
kds, conc_sec = column_section(column_type="tie")
conc_sec.plot_section()

/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53080 (\N{HANGUL SYLLABLE KON}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53356 (\N{HANGUL SYLLABLE KEU}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 47532 (\N{HANGUL SYLLABLE RI}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/tmp/claude-0/-home-user-wkclauderepositoty/78f995f2-8acb-52ee-a7a0-dbb2b3510f36/scratchpad/venv312/lib/python3.12/site-packages/concreteproperties/post.py:90: UserWarning: Glyph 53944 (\N{HANGUL SYLLABL

<Axes: title={'center': 'Reinforced Concrete Section'}>

In [4]:
n_max_nom, n_max_des = kds.max_axial_strength()

print(f"공칭 축강도       Po         = {kds.squash_load / 1e3:10,.1f} kN")
print(f"최대 공칭 축강도  0.80*Po    = {n_max_nom / 1e3:10,.1f} kN")
print(f"최대 설계 축강도  phi*Pn,max = {n_max_des / 1e3:10,.1f} kN")
print(f"공칭 인장강도     Pnt        = {kds.tensile_load / 1e3:10,.1f} kN")

공칭 축강도       Po         =    6,905.1 kN
최대 공칭 축강도  0.80*Po    =    5,524.1 kN
최대 설계 축강도  phi*Pn,max =    3,590.7 kN
공칭 인장강도     Pnt        =   -1,238.7 kN


## 상관도 생성

`moment_interaction_diagram` 은 (설계 상관도, 공칭 상관도, phi 목록) 을
반환한다.

In [5]:
f_mi, mi, phis = kds.moment_interaction_diagram(
    n_points=24, progress_bar=False
)

print(f"{'Nn(kN)':>10} {'Mn(kNm)':>10} {'et':>10} {'분류':>10}"
      f" {'phi':>7} {'phiN(kN)':>10} {'phiM(kNm)':>11}")
print("-" * 76)
for r_u, r_f, p in zip(mi.results, f_mi.results, phis, strict=True):
    e = kds.net_tensile_strain(theta=0, d_n=r_u.d_n)
    e_str = f"{'inf':>10}" if e == float("inf") else f"{e:10.5f}"
    print(
        f"{r_u.n / 1e3:10.1f} {r_u.m_x / 1e6:10.1f} {e_str}"
        f" {kds.section_classification(eps_t=e):>10} {p:7.3f}"
        f" {r_f.n / 1e3:10.1f} {r_f.m_x / 1e6:11.1f}"
    )

    Nn(kN)    Mn(kNm)         et         분류     phi   phiN(kN)   phiM(kNm)
----------------------------------------------------------------------------
    5524.1        0.0   -0.00330     압축지배단면   0.650     3590.7         0.0
    5359.1      294.6   -0.00040     압축지배단면   0.650     3483.4       191.5
    5152.5      323.8   -0.00029     압축지배단면   0.650     3349.1       210.5
    4943.0      350.9   -0.00017     압축지배단면   0.650     3213.0       228.1
    4730.3      376.0   -0.00004     압축지배단면   0.650     3074.7       244.4
    4513.9      398.9    0.00010     압축지배단면   0.650     2934.0       259.3
    4293.4      419.9    0.00026     압축지배단면   0.650     2790.7       273.0
    4068.0      439.1    0.00043     압축지배단면   0.650     2644.2       285.4
    3837.2      456.4    0.00062     압축지배단면   0.650     2494.2       296.7
    3600.0      472.1    0.00082     압축지배단면   0.650     2340.0       306.8
    3355.4      486.2    0.00106     압축지배단면   0.650     2181.0       316.0
    3109.6      499.0  

In [6]:
from concreteproperties.results import MomentInteractionResults

fig, ax = plt.subplots(figsize=(6.5, 5))
MomentInteractionResults.plot_multiple_diagrams(
    [mi, f_mi], ["nominal (Mn, Pn)", "design (phi*Mn, phi*Pn)"],
    fmt="-", ax=ax, render=False,
)
ax.axhline(n_max_des / 1e3 * 1e3, ls="--", color="grey", lw=0.8)
ax.set_title("P-M interaction diagram (KDS 14 20)")
ax.grid(alpha=0.3)

설계 상관도가 공칭 상관도 안쪽에 있고, 압축측이 최대 설계 축강도에서
절단된다. 두 곡선의 간격이 일정하지 않은 것은 $\phi$ 가 축력에 따라
0.65 에서 0.85 까지 변하기 때문이다.

## 축력에 따른 강도감소계수

In [7]:
n_list = np.array([r.n for r in mi.results]) / 1e3
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(n_list, phis, "o-")
ax.set_xlabel("nominal axial force, Pn (kN)")
ax.set_ylabel("phi")
ax.set_title("Strength reduction factor along the diagram")
ax.grid(alpha=0.3)

## 설계 축력별 설계 휨강도

In [8]:
print(f"{'Nd(kN)':>10} {'phi':>8} {'et':>10} {'분류':>10} {'phiMn(kNm)':>12}")
print("-" * 56)
for n_d in [-800, -400, 0, 400, 800, 1200, 1600, 2000, 2800, 3400]:
    f_res, u_res, phi = kds.ultimate_bending_capacity(n_design=n_d * 1e3)
    e = kds.net_tensile_strain(theta=0, d_n=u_res.d_n)
    e_str = f"{'inf':>10}" if e == float("inf") else f"{e:10.5f}"
    print(
        f"{n_d:10.0f} {phi:8.3f} {e_str}"
        f" {kds.section_classification(eps_t=e):>10}"
        f" {f_res.m_x / 1e6:12.1f}"
    )

    Nd(kN)      phi         et         분류   phiMn(kNm)
--------------------------------------------------------


      -800    0.850        inf     인장지배단면         52.1


      -400    0.850        inf     인장지배단면        134.6


         0    0.850    0.01675     인장지배단면        217.0


       400    0.850    0.01063     인장지배단면        290.3


...

      1200    0.828    0.00467     변화구간단면        386.2


      1600    0.665    0.00222     변화구간단면        345.6


      2000    0.650    0.00135     압축지배단면        325.4


      2800    0.650    0.00025     압축지배단면        272.1


      3400    0.650   -0.00033     압축지배단면        203.5


## 띠철근과 나선철근 비교

나선철근 기둥은 압축지배단면의 $\phi$ 가 0.70 이고 최대 축강도 저감계수도
0.85 라 상관도가 바깥쪽에 놓인다.

In [9]:
kds_s, _ = column_section(column_type="spiral")
f_mi_s, _, _ = kds_s.moment_interaction_diagram(
    n_points=24, progress_bar=False
)

fig, ax = plt.subplots(figsize=(6.5, 5))
MomentInteractionResults.plot_multiple_diagrams(
    [f_mi, f_mi_s], ["tie", "spiral"], fmt="-", ax=ax, render=False
)
ax.set_title("Design diagram: tie vs spiral")
ax.grid(alpha=0.3)

## 설계 단면력 판정

In [10]:
for n_d, m_d in [(1500e3, 300e6), (1500e3, 450e6), (3000e3, 200e6)]:
    inside = f_mi.point_in_diagram(n=n_d, m=m_d)
    print(f"Nd = {n_d / 1e3:7.0f} kN, Md = {m_d / 1e6:6.0f} kN.m"
          f"  ->  {'안전' if inside else '불안전'}")

Nd =    1500 kN, Md =    300 kN.m  ->  안전
Nd =    1500 kN, Md =    450 kN.m  ->  불안전
Nd =    3000 kN, Md =    200 kN.m  ->  안전
